<a href="https://colab.research.google.com/github/Alfonsotomax/Adfonfin/blob/main/notebook/AVE_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
import torch
import numpy as np
import torchvision.transforms.v2 as T
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
from google.colab import drive
from torch.utils.data import TensorDataset, DataLoader


# Runtime → Change runtime type → T4 GPU for faster training compared to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drive.mount('/content/drive')

!cp '/content/drive/MyDrive/dog-project-data/processed/dogs64.npy' '/content/'
df = np.load('/content/dogs64.npy')
csv_data = pd.read_csv('/content/drive/MyDrive/dog-project-data/processed/metadata.csv')





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# Getting the index of the training data & val data
train = csv_data.index[csv_data["split"] == "train"].tolist()
val   = csv_data.index[csv_data["split"] == "val"].tolist()

# getting the actual images from the indexs
train_images = df[train]
val_images = df[val]

del df # free up memory

# Converting the 0...255 images to float 32
#train_images = train_images.astype(np.float32)
#val_images = val_images.astype(np.float32)

# mapping 0..255 to [-1,1] for tanh
#train_images = (train_images / 127.5) - 1
#val_images = (val_images / 127.5) - 1


In [37]:
#print(train_images.shape) #(72473, 64, 64, 3) need (N, 3, 64, 64)
train_images = train_images.transpose(0, 3, 1, 2)
val_images = val_images.transpose(0, 3, 1, 2)
#print(val_images.shape)

train_images = np.ascontiguousarray(train_images)
val_images = np.ascontiguousarray(val_images)

# convert to tensor from numpy array
train_images = torch.from_numpy(train_images)
val_images = torch.from_numpy(val_images)


In [38]:
# Need TensorDataset
train_dataset = TensorDataset(train_images)
val_dataset = TensorDataset(val_images)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last = True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, drop_last = False)



#(a, ) = next(iter(train_loader))
#a = a.to(device)
#a = a.float() / 127.5 - 1.0

def prep_batch(self, batch):
  return (batch.to(device, non_blocking=True).float / 127.5) -1.0

#print(a.shape)
#print(a.dtype)
#print(a.device)
#print(a.min())
#print(a.max())

#for (batch, ) in train_loader:
  #batch = batch.to(device)

In [39]:
def show_grid(batch):
  data = batch[:64]
  rescale = (data + 1) * 127.5
  rescale = rescale.clamp(0,255)
  rescale = rescale.byte()
  rescale = rescale.permute(0,2,3,1)
  rescale = rescale.detach().cpu().numpy()
  grid = rescale.reshape(8, 8, 64, 64, 3).transpose(0, 2, 1, 3, 4).reshape(8 * 64, 8 * 64, 3)
  plt.figure(figsize=(10, 10))
  plt.imshow(grid)
  plt.axis('off')
  plt.show()

In [40]:
class Encoder(nn.Module):
  # runs once when you create the Neural Network architecture
  def __init__(self, latent_dim):
    super().__init__()
    self.convs = nn.Sequential(
      # 3x64x64 -> 32x32x32
      nn.Conv2d(3, 32, 4, stride=2, padding=1),
      nn.BatchNorm2d(32),
      nn.LeakyReLU(0.2),

      # 32x32x32 -> 64x16x16
      nn.Conv2d(32, 64, 4, stride=2, padding=1),
      nn.BatchNorm2d(64),
      nn.LeakyReLU(0.2),

      # 64x16x16 -> 128x8x8
      nn.Conv2d(64, 128, 4, stride=2, padding=1),
      nn.BatchNorm2d(128),
      nn.LeakyReLU(0.2),

      # 128x8x8 -> 256x4x4
      nn.Conv2d(128, 256, 4, stride=2, padding=1),
      nn.BatchNorm2d(256),
      nn.LeakyReLU(0.2),
    )
    self.fc_logvar = nn.Linear(in_features=256 * 4 * 4, out_features=latent_dim) # learning for the log variance of the latent space
    self.fc_mu = nn.Linear(in_features=256 * 4 * 4, out_features=latent_dim) # learning for the mean of the latent space

  # runs on every batch
  def forward(self, x):
    x = self.convs(x)
    x = torch.flatten(x, start_dim=1)
    mu = self.fc_mu(x)
    logvar = self.fc_logvar(x)
    return mu, logvar


In [45]:
class Decoder(nn.Module):
  def __init__(self, latent_dim):
    super().__init__()
    # z (128 numbers) -> 4096 numbers, which get reshaped to 256x4x4
    self.fc = nn.Linear(in_features=latent_dim, out_features=256 * 4 * 4)

    self.deconvs = nn.Sequential(
      # 256x4x4 -> 128x8x8
      nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
      nn.BatchNorm2d(128),
      nn.ReLU(),

      # 128x8x8 -> 64x16x16
      nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
      nn.BatchNorm2d(64),
      nn.ReLU(),

      # 64x16x16 -> 32x32x32
      nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(),

      # 32x32x32 -> 3x64x64  (last layer: no BatchNorm, Tanh for [-1, 1])
      nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
      nn.Tanh(),
    )

  def forward(self, z):
    x = self.fc(z)               # (B, 128) -> (B, 4096)
    x = x.view(-1, 256, 4, 4)    # (B, 4096) -> (B, 256, 4, 4)
    x = self.deconvs(x)          # (B, 256, 4, 4) -> (B, 3, 64, 64)
    return x

In [42]:
class VAE(nn.Module):
  def __init__(self, latent_dim):
    super().__init__()
    self.latent_dim = latent_dim
    self.encoder = Encoder(latent_dim)
    self.decoder = Decoder(latent_dim)

  def forward(self, x):
    mu, logvar = self.encoder(x)
    z = self.reparameterize(mu, logvar)
    recon = self.decoder(z)
    return recon, mu, logvar

  def reparameterize(self, mu, logvar):
    variance = torch.exp(0.5 * logvar)
    epsilon = torch.randn_like(variance)
    return mu + variance * epsilon

  @torch.no_grad
  def sample(self, n):
    z = torch.randn(n, self.latent_dim, device=next(self.parameters()).device)
    return self.decode(z)


In [43]:
def VAE_loss(recon, x, mu, logvar, beta=1.0):
  batch_size = x.size(0)

  # Mean square error as the loss function
  recon_loss = F.mse_loss(recon, x, reduction="sum") / batch_size

  # Kl is the divergence between N(mu, var^2) and N(0,1)
  kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size

  total = recon_loss + beta * kl
  return total, recon_loss, kl